# Workbook Upload Demo

This notebook mirrors `scripts/upload_results_example_data.py`.

It parses the example workbook, builds validated analyses and results, and optionally uploads the rows that are compatible with the current dev backend.

In [ ]:
from __future__ import annotations

import enum
import math
import os
import sys
import pathlib
from pathlib import Path
from tqdm.notebook import tqdm as tqdm_notebook

import pandas as pd
from IPython.display import display

if not hasattr(enum, "StrEnum"):
    class _CompatStrEnum(str, enum.Enum):
        pass
    enum.StrEnum = _CompatStrEnum

workspace_root = Path.cwd().resolve().parent
sys.path.insert(0, str(workspace_root / "src"))
sys.path.insert(0, str(workspace_root.parent / "owi-metadatabase-sdk" / "src"))

from owi.metadatabase.locations.io import LocationsAPI
from owi.metadatabase.results import (
    AnalysisDefinition,
    LifetimeDesignFrequencies,
    LifetimeDesignVerification,
    ResultsAPI,
    WindSpeedHistogram,
)
from owi.metadatabase.results.serializers import DjangoAnalysisSerializer, DjangoResultSerializer

DEFAULT_BASE_URL = "https://owimetadatabase-dev.azurewebsites.net/api/v1"
BASE_PATH = Path().parent
DEFAULT_WORKBOOK = BASE_PATH / "data" / "results-example-data.xlsx"
print(f"Using workbook at {pathlib.Path(DEFAULT_WORKBOOK).resolve()}")
TOKEN_ENV_VAR = "OWI_METADATABASE_API_TOKEN"


In [ ]:
BASE_URL = DEFAULT_BASE_URL
WORKBOOK = DEFAULT_WORKBOOK
PROJECTSITE = "Belwind"
TOKEN = ""  # os.getenv(TOKEN_ENV_VAR)
LIVE_UPLOAD = TOKEN is not None

print({"workbook": str(WORKBOOK), "projectsite": PROJECTSITE, "live_upload": LIVE_UPLOAD})


In [ ]:
def resolve_site_and_locations(api_root: str, token: str | None, projectsite: str, turbines: set[str], live: bool):
    if not live or token is None:
        return 1, {turbine: None for turbine in turbines}
    locations_api = LocationsAPI(api_root=api_root, token=token)
    site_id = int(locations_api.get_projectsite_detail(projectsite=projectsite)["id"])
    assetlocations = locations_api.get_assetlocations(projectsite=projectsite)["data"]
    resolved_locations = {
        str(row["title"]): int(row["id"])
        for row in assetlocations.to_dict(orient="records")
        if row.get("title") is not None and row.get("id") is not None
    }
    mapping = {turbine: resolved_locations.get(turbine) for turbine in sorted(turbines)}
    return site_id, mapping


# def parse_histogram_sheet(path: Path, site_id: int, location_map: dict[str, int | None]):
#     frame = pd.read_excel(path, sheet_name="Lifetime - Wind Histogram", header=1)
#     bin_columns = [column for column in frame.columns if isinstance(column, str) and column.startswith("[")]
#     series = []
#     for row in frame.to_dict(orient="records"):
#         scope = str(row["Scope"])
#         parsed_bins = []
#         for column in bin_columns:
#             left, right = column.removeprefix("[").removesuffix("[").split(",")
#             parsed_bins.append((float(left), float(right)))
#         series.append(
#             {
#                 "title": row["Title"],
#                 "description": row["Description"],
#                 "scope_label": scope,
#                 "site_id": site_id,
#                 "location_id": location_map.get(scope),
#                 "bins": parsed_bins,
#                 "values": [float(row[column]) for column in bin_columns],
#                 "metadata": {"scope_label": scope},
#             }
#         )
#     analysis = AnalysisDefinition(
#         name="Lifetime Wind Histogram",
#         source_type="notebook",
#         description="Workbook upload for the lifetime wind histogram example sheet.",
#         additional_data={"sheet_name": "Lifetime - Wind Histogram"},
#     )
#     return analysis, WindSpeedHistogram().to_results({"series": series})


# def parse_verification_sheet(path: Path, site_id: int, location_map: dict[str, int | None]):
#     frame = pd.read_excel(path, sheet_name="Lifetime -  Design verification")
#     rows = []
#     for row in frame.to_dict(orient="records"):
#         rows.append(
#             {
#                 "timestamp": row["timestamp"],
#                 "turbine": str(row["Turbine"]),
#                 "FA1": row.get("FA1 (Hz)"),
#                 "SS1": row.get("SS1 (Hz)"),
#                 "SS2": row.get("SS2 (Hz)"),
#                 "FA2": row.get("FA2 (Hz)"),
#                 "site_id": site_id,
#                 "location_id": location_map.get(str(row["Turbine"])),
#             }
#         )
#     analysis = AnalysisDefinition(
#         name="Lifetime Design Verification",
#         source_type="notebook",
#         description="Workbook upload for the lifetime design verification sheet.",
#         additional_data={"sheet_name": "Lifetime -  Design verification"},
#     )
#     return analysis, LifetimeDesignVerification().to_results({"rows": rows})


def parse_frequencies_sheet(path: Path, site_id: int, location_map: dict[str, int | None]):
    frame = pd.read_excel(path, sheet_name="Lifetime -  Design frequencies")
    rows = []
    for row in frame.to_dict(orient="records"):
        rows.append(
            {
                "turbine": str(row["Turbine"]),
                "reference": row["Reference"],
                "FA1": row.get("FA1 [Hz]"),
                "SS1": row.get("SS1 [Hz]"),
                "FA2": row.get("FA2 [Hz]"),
                "SS2": row.get("SS2 [Hz]"),
                "site_id": site_id,
                "location_id": location_map.get(str(row["Turbine"])),
            }
        )
    analysis = AnalysisDefinition(
        name="Lifetime Design Frequencies",
        source_type="notebook",
        description="Workbook upload for the lifetime design frequencies sheet.",
        additional_data={"sheet_name": "Lifetime -  Design frequencies"},
    )
    return analysis, LifetimeDesignFrequencies().to_results({"rows": rows})


In [ ]:
verification_frame = pd.read_excel(WORKBOOK, sheet_name="Lifetime -  Design verification")
frequencies_frame = pd.read_excel(WORKBOOK, sheet_name="Lifetime -  Design frequencies")
turbines = set(verification_frame["Turbine"].astype(str)) | set(frequencies_frame["Turbine"].astype(str))
site_id, location_map = resolve_site_and_locations(BASE_URL, TOKEN, PROJECTSITE, turbines, LIVE_UPLOAD)

display(pd.DataFrame([{"turbine": turbine, "location_id": location_map[turbine]} for turbine in sorted(location_map)]).head())


In [ ]:
payload_sets = [
    # parse_histogram_sheet(WORKBOOK, site_id, location_map),
    # parse_verification_sheet(WORKBOOK, site_id, location_map),
    parse_frequencies_sheet(WORKBOOK, site_id, location_map),
]

summary_rows = [
    {
        "analysis": analysis_definition.name,
        "results": len(result_series),
        "sheet": analysis_definition.additional_data["sheet_name"],
    }
    for analysis_definition, result_series in payload_sets
]

display(pd.DataFrame(summary_rows))


In [ ]:
analysis_serializer = DjangoAnalysisSerializer()
result_serializer = DjangoResultSerializer()

preview_rows = []
for analysis_definition, result_series in payload_sets:
    result_payloads = [result_serializer.to_payload(item, analysis_id=0) for item in result_series]
    invalid_payloads = [payload for payload in result_payloads if payload.get("location") is None]
    valid_payloads = [payload for payload in result_payloads if payload.get("location") is not None]
    preview_rows.append(
        {
            "analysis": analysis_definition.name,
            "uploadable_rows": len(valid_payloads),
            "skipped_rows": len(invalid_payloads),
            "first_result": result_series[0].short_description,
        }
    )

display(pd.DataFrame(preview_rows))


## Optional Live Upload

The next cell uploads only the rows that can be resolved to a backend `location` and have finite numeric values.

In [ ]:
def is_finite_payload(payload: dict) -> bool:
    value_keys = ["value_col1", "value_col2", "value_col3"]
    for key in value_keys:
        values = payload.get(key)
        if values is None:
            continue
        if any(not math.isfinite(float(value)) for value in values):
            return False
    return True


if not LIVE_UPLOAD:
    print("Dry-run mode: set OWI_METADATABASE_API_TOKEN to enable uploads.")
else:
    api = ResultsAPI(api_root=BASE_URL, token=TOKEN)
    upload_log = []
    for analysis_definition, result_series in payload_sets:
        result_payloads = [result_serializer.to_payload(item, analysis_id=0) for item in result_series]
        valid_payloads = [payload for payload in result_payloads if payload.get("location") is not None and is_finite_payload(payload)]
        if not valid_payloads:
            upload_log.append({"analysis": analysis_definition.name, "uploaded_rows": 0, "status": "skipped"})
            continue

        created_analysis = api.create_analysis(analysis_serializer.to_payload(analysis_definition))
        analysis_id = int(created_analysis["id"])
        upload_payloads = [{**payload, "analysis": analysis_id} for payload in valid_payloads]
        # create results not bulk:

        for payload in tqdm_notebook(upload_payloads,
                                     desc=f"Uploading results for {analysis_definition.name}"):
            response = api.create_result(payload)
        # response = api.create_results_bulk(upload_payloads)
            upload_log.append({"analysis": analysis_definition.name, "uploaded_rows": len(upload_payloads), "status": bool(response["exists"])})

    display(pd.DataFrame(upload_log))


## Retrieve And Plot Uploaded Lifetime Design Frequencies

The next cell fetches the uploaded `Lifetime Design Frequencies` analysis rows back from the backend, reconstructs the normalized result table with the results package, and renders package plots from the retrieved rows.

In [ ]:
from owi.metadatabase.results import plot_lifetime_design_frequencies_by_location
from owi.metadatabase.results.analyses.lifetime_design_frequencies import LifetimeDesignFrequencies

backend_analysis_name = "Lifetime Design Frequencies"

retrieval_api = api if "api" in globals() else ResultsAPI(api_root=BASE_URL, token=TOKEN)
analyses_frame = retrieval_api.list_analyses(name=backend_analysis_name)["data"]
raw_results_frame = retrieval_api.get_results(projectsite="non-existing-project")["data"]
raw_results_frame.head()


In [ ]:

sanitized_rows = []
for row in raw_results_frame.to_dict(orient="records"):
    sanitized_row = dict(row)
    if sanitized_row.get("value_col3") == []:
        sanitized_row["value_col3"] = None
        sanitized_row["name_col3"] = None
        sanitized_row["units_col3"] = None
    sanitized_rows.append(sanitized_row)

retrieved_series = [
    result_serializer.from_mapping(row)
    for row in sanitized_rows
]
frequencies_analysis = LifetimeDesignFrequencies()
retrieved_frequencies = frequencies_analysis.from_results(retrieved_series)

location_lookup = pd.DataFrame(
    [
        {"id": location_id, "title": turbine}
        for turbine, location_id in sorted(location_map.items())
        if location_id is not None
    ]
)

comparison_plot = frequencies_analysis.plot(retrieved_series)
location_plot = plot_lifetime_design_frequencies_by_location(
    retrieved_frequencies,
    location_frame=location_lookup if not location_lookup.empty else None,
)

# display(retrieved_frequencies)
display(comparison_plot.chart.render_notebook())
# display(location_plot.notebook)
